# Environment Setup

This notebook verifies your OpenShift environment, and configures environment variables.

> **Assumption:** All operators and infrastructure (including MaaS with PostgreSQL) are already installed via [RHOAI-Toolkit](https://github.com/hyogrin/RHOAI-Toolkit). This notebook only verifies and configures application-level resources.

## 0. Bootstrap (Workbench + Local)

This cell prepares the execution environment. It auto-detects whether you are running in an **RHOAI Workbench** or on a **local machine** and handles dependencies accordingly.

| Environment | `oc` CLI | Python deps | Auth |
|-------------|----------|-------------|------|
| **Local machine** | Pre-installed | `uv sync` (see README) | `oc login -u <user>` |
| **RHOAI Workbench** | Auto-installed below | Auto-installed via `pip` | `oc login -u <user>` in Workbench terminal |

> **Important:** If running in a Workbench, open the Workbench **terminal** first and run:
> ```
> oc login -u <username> https://api.<cluster>:6443
> ```
> Use your **own user credentials** (not the pod's ServiceAccount token) so you have admin-group privileges.

In [ ]:
import shutil, subprocess, os, sys

IN_WORKBENCH = os.path.exists("/opt/app-root") or bool(os.getenv("JUPYTER_IMAGE"))

print(f"Environment: {'RHOAI Workbench' if IN_WORKBENCH else 'Local machine'}")
print("=" * 50)

# --- 1) Install oc CLI if missing (Workbench only) ---
if not shutil.which("oc"):
    if IN_WORKBENCH:
        print("Installing oc CLI...")
        subprocess.run([
            "bash", "-c",
            "curl -sL https://mirror.openshift.com/pub/openshift-v4/clients/ocp/stable/"
            "openshift-client-linux.tar.gz | tar xz -C /opt/app-root/bin oc kubectl"
        ], check=True)
        print(f"  oc installed: {shutil.which('oc')}")
    else:
        print("❌ oc CLI not found. Install it before proceeding.")
        print("   https://mirror.openshift.com/pub/openshift-v4/clients/ocp/stable/")
else:
    r = subprocess.run(["oc", "version", "--client", "-o", "json"], capture_output=True, text=True)
    ver = r.stdout.strip()[:60] if r.returncode == 0 else "unknown"
    print(f"✅ oc CLI: {ver}")

# --- 2) Install Python dependencies (Workbench only) ---
if IN_WORKBENCH:
    missing = []
    for pkg in ["dotenv", "openai", "httpx", "requests"]:
        try:
            __import__(pkg)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing Python dependencies (missing: {missing})...")
        subprocess.run([
            sys.executable, "-m", "pip", "install", "-q",
            "python-dotenv>=1.0", "openai>=1.0", "httpx>=0.27", "requests>=2.31",
            "eval-hub-sdk==0.1.8", "mlflow==3.13.0",
        ], check=True)
        print("  Dependencies installed.")
    else:
        print("✅ Python dependencies already installed.")
else:
    print("✅ Local environment (use 'uv sync' for dependencies).")

# --- 3) Verify cluster access ---
r = subprocess.run(["oc", "whoami"], capture_output=True, text=True)
if r.returncode == 0:
    user = r.stdout.strip()
    server = subprocess.run(["oc", "whoami", "--show-server"], capture_output=True, text=True).stdout.strip()
    print(f"✅ Logged in as: {user}")
    print(f"   Cluster: {server}")
else:
    print("❌ Not logged in to cluster.")
    if IN_WORKBENCH:
        print("   Open Workbench terminal and run:")
        print("     oc login -u <username> https://api.<cluster>:6443")
    else:
        print("   Run: oc login -u <username> https://api.<cluster>:6443")

## 1. Verify Infrastructure

One check per topic area. All operators/components should already be installed via [RHOAI-Toolkit](https://github.com/hyogrin/RHOAI-Toolkit) Option 3.

In [ ]:
import subprocess, json

def _run(cmd, timeout=10):
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    return r.returncode == 0, r.stdout.strip()

def _csv_exists(pattern, namespace=""):
    ns_flag = ["-n", namespace] if namespace else ["-A"]
    ok, out = _run(["oc", "get", "csv", *ns_flag, "--no-headers"])
    return ok and pattern in out

print("=" * 60)
print("  Infrastructure Verification (one check per area)")
print("=" * 60)

# 1. Cluster
ok, user = _run(["oc", "whoami"])
_, server = _run(["oc", "whoami", "--show-server"])
print(f"{'✅' if ok else '❌'} Cluster: {user} @ {server}")

# 2. RHOAI Operator
ok = _csv_exists("rhods", "redhat-ods-operator")
print(f"{'✅' if ok else '❌'} RHOAI Operator")

# 3. GPU Nodes
ok, out = _run(["oc", "get", "nodes", "-l", "nvidia.com/gpu.present=true", "--no-headers"])
gpu_count = len(out.splitlines()) if ok and out else 0
print(f"{'✅' if gpu_count > 0 else '⚠️ '} GPU Nodes: {gpu_count}")

# 4. MaaS Gateway
ok, _ = _run(["oc", "get", "gateway", "maas-default-gateway", "-n", "openshift-ingress"])
print(f"{'✅' if ok else '❌'} MaaS Gateway")

# 5. MaaS Tenant
ok, out = _run(["oc", "get", "tenants.maas.opendatahub.io", "default-tenant",
                "-n", "models-as-a-service", "-o", "jsonpath={.status.conditions[?(@.type==\"Ready\")].status}"])
print(f"{'✅' if out == 'True' else '⚠️ '} MaaS Tenant: {'Ready' if out == 'True' else out or 'not found'}")

# 6. EvalHub
ok, out = _run(["oc", "get", "evalhub", "evalhub", "-n", "redhat-ods-applications",
                "-o", "jsonpath={.status.phase}"])
print(f"{'✅' if out == 'Ready' else '⚠️ '} EvalHub: {out or 'not found'}")

# 7. Observability (COO)
ok = _csv_exists("cluster-observability-operator", "openshift-operators")
print(f"{'✅' if ok else '⚠️ '} Observability (COO)")

print("=" * 60)

## 2. Configure Environment Variables

In [ ]:
from pathlib import Path

env_path = Path("../.env")
sample_path = Path("../sample.env")

if not env_path.exists():
    if sample_path.exists():
        env_path.write_text(sample_path.read_text())
        print(f"Created {env_path} from sample.env")
        print("⚠️  Edit .env and fill in your tokens before proceeding.")
    else:
        print("❌ sample.env not found.")
else:
    print(f"✅ {env_path} already exists.")

## 3. Next Steps

1. **Deploy Model** → `2_model_deploy.ipynb` — deploy a model via RHOAI Dashboard
2. **Phase 1** → `../1_mcp_servers/2_deploy_mcp_servers.ipynb` — verify MCP tool servers
3. **Phase 2** → `../2_maas/2_enable_maas.ipynb` — register models, create API keys
4. **Phase 3** → `../3_basic_run/1_ide_model_config.ipynb` — configure IDE and run coding assistant